# 05 — Agentic AI Prototype
End-to-end walkthrough of the multi-agent investment pipeline:
MarketMonitor → ForecastAgent → DecisionAgent → RebalanceAgent

In [ ]:
import sys
sys.path.insert(0, '..')

import asyncio
import json
import logging

import pandas as pd

logging.basicConfig(level=logging.INFO, format='%(asctime)s [%(levelname)s] %(message)s')
print('Setup complete.')

## 1. Individual Agent Walkthrough
Instantiate and inspect each agent before wiring them together.

In [ ]:
from ml.agents.market_monitor import MarketMonitorAgent

monitor = MarketMonitorAgent(threshold=0.005, poll_interval=60)
print('MarketMonitor status:', monitor.status)
print('Alert threshold:',      monitor.threshold)
print('Poll interval (s):',    monitor.poll_interval)

In [ ]:
from ml.agents.forecast_agent import ForecastAgent

forecaster = ForecastAgent()
print('ForecastAgent status:', forecaster.status)

# Run the forecaster synchronously via asyncio.run for notebook convenience
forecasts = asyncio.run(forecaster.run())
if forecasts:
    rows = [
        {
            'Pair': pair,
            'Last Actual': round(data['last_actual'], 6),
            'Predicted Next': round(data['predicted_next'], 6),
            'Forecast Return %': f"{data['forecast_return']*100:.3f}%",
        }
        for pair, data in forecasts.items()
    ]
    display(pd.DataFrame(rows).set_index('Pair'))
else:
    print('No forecasts available. Run training first: python ml/training/train.py')

In [ ]:
from ml.agents.decision_agent import DecisionAgent

decision_agent = DecisionAgent(
    budget=10_000.0,
    risk_tolerance='medium',
    min_weight=0.05,
    max_weight=0.60,
)

if forecasts:
    decision = asyncio.run(decision_agent.run(forecasts))
    print('Rebalance needed:', decision['rebalance_needed'])
    print('Expected return: ', f"{decision['expected_return_pct']*100:.2f}%")
    print('Allocations:')
    for pair, weight in decision['allocations'].items():
        amount = decision['amounts'][pair]
        print(f'  {pair}: {weight*100:.1f}%  =  ${amount:,.2f}')

In [ ]:
from ml.agents.rebalance_agent import RebalanceAgent

rebalancer = RebalanceAgent(simulation=True)

if forecasts:
    rebalance_result = asyncio.run(rebalancer.run(decision))
    print('Rebalanced:', rebalance_result.get('rebalanced'))
    print('Trades:')
    for pair, info in rebalance_result.get('trades', {}).items():
        print(f'  {pair}: {info}')

## 2. Full Orchestrated Pipeline
Run all agents through the AgentOrchestrator in a single call.

In [ ]:
from ml.agents.agent_orchestrator import AgentOrchestrator

orchestrator = AgentOrchestrator(
    budget=10_000.0,
    risk_tolerance='medium',
    simulation=True,
)

summary = asyncio.run(orchestrator.run_pipeline(trigger='manual'))
print(json.dumps(summary, indent=2))

## 3. Pipeline Log Inspection
Review the combined event log produced by all agents.

In [ ]:
log = orchestrator.get_full_log()
log_df = pd.DataFrame(log)
if not log_df.empty:
    display(log_df[['timestamp', 'agent', 'action', 'detail']].tail(20))
else:
    print('Log is empty (pipeline did not produce events).')

## 4. Alert-Driven Mode (Simulated)
Simulate a market shift that triggers the pipeline automatically.

In [ ]:
# Inject a synthetic price shift above the alert threshold (0.5% log-return)
synthetic_prices = {'EUR_USD': 1.12, 'AUD_USD': 0.68, 'NZD_USD': 0.63}
orchestrator.monitor.last_prices = {'EUR_USD': 1.10, 'AUD_USD': 0.70, 'NZD_USD': 0.64}

alerts = orchestrator.monitor.detect_shifts(synthetic_prices)
print('Alerts detected:', alerts)

if alerts:
    print('\nTriggering pipeline via alert ...')
    alert_summary = asyncio.run(orchestrator.run_pipeline(trigger='alert'))
    print('Alert pipeline result:', alert_summary.get('rebalanced'))

## 5. Agent Status Report
Inspect the status of every agent after a full run.

In [ ]:
status = orchestrator.status_report()
pd.DataFrame([status]).T.rename(columns={0: 'Status'})

## 6. Budget & Risk Sensitivity
Compare pipeline outputs across different budgets and risk tolerance levels.

In [ ]:
import logging
logging.disable(logging.CRITICAL)

rows = []
for budget in [5_000, 10_000, 50_000]:
    for risk in ['low', 'medium', 'high']:
        orch = AgentOrchestrator(budget=budget, risk_tolerance=risk, simulation=True)
        try:
            result = asyncio.run(orch.run_pipeline(trigger='manual'))
            rows.append({
                'Budget': f'${budget:,}',
                'Risk': risk.title(),
                'Expected Return': result.get('expected_return', 'N/A'),
                'Rebalanced': result.get('rebalanced', False),
            })
        except Exception as exc:
            rows.append({'Budget': f'${budget:,}', 'Risk': risk.title(),
                         'Expected Return': 'ERROR', 'Rebalanced': str(exc)[:40]})

logging.disable(logging.NOTSET)
pd.DataFrame(rows).set_index(['Budget', 'Risk'])